In [1]:
import os
import random
import json
import asyncio
import nest_asyncio
from datetime import datetime
from openai import AsyncOpenAI
from tqdm.notebook import tqdm
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
timestamp

'20250721_1450'

In [3]:
nest_asyncio.apply()

In [4]:
client = AsyncOpenAI()

In [5]:
semaphore = asyncio.Semaphore(5)

In [6]:
# model="gpt-4o-mini"
# model="o3-mini"
# model="o4-mini"
# model="gpt-4.1"
model="gpt-4o"

In [19]:
async def generate_questions_for_file(filepath: str, n_questions: int = 2) -> list[dict]:
    """
    Generate N quqesion-answer pairs for a given event file using gpt-4.1
    returns a list of dicts with question, answer and filename
    """
    async with semaphore:
        with open(filepath, 'r', encoding='utf-8') as f:
            lines = f.readlines()
        
        original_title = lines[1].strip('# ').strip()
        clean_content = "".join(lines[2:])
        filename = os.path.basename(filepath)

        # prompt = f"""Given the following event file content, generate {n_questions} query-answer pairs.
        #             Current date is: {datetime.now().date().isoformat()}
        #             Requirements:
        #             - Queries should be one sentence search queries looking for events
        #                 - They should be framed as a general search intent (the query reader will not see the event file OR title OR any other information about the event)
        #                 - Queries should be what a user would type when searching for this type of event without knowing it exists
        #                 - Include event type, time period and the location
        #                 - If the event date is before current date, the question must narrow down the time period to e.g. "drugi tydzień Lipca" or "w weekend 5-6 lipca"
                    
        #             - Answers should contain the full event details: title, date, location
        #             - All queries and answers must be in Polish language
        #             - Return as a JSON object with a "questions" list containing dicts with "question" and "answer" fields

        #             Event content:
        #             {content}

        #             Schema: 
        #             {{
        #                 "questions": [
        #                     {{
        #                         "question": "search query text",
        #                         "answer": "Title: [event title]\\nDate: [date and time]\\nLocation: [location]"
        #                     }},
        #                     ...
        #                 ]
        #             }}

        #             Return ONLY the JSON object, no other text."""

        # prompt = f"""You are an expert Test Data Engineer creating a challenging and realistic test set for an AI event search agent.
        #             Given the following event file content, generate {n_questions} query-answer pairs.
        #             **Event Content to Analyze:**
        #             '''
        #             {clean_content}
        #             '''
        #             Current date is: {datetime.now().date().isoformat()}

        #             ---
        #             **QUERY GENERATION RULES (You must follow all rules):**

        #             1.  **Semantic Realism (No Cheating):** Your primary goal is to create queries that a user who **does not know the event's title** would type. You must infer the event's topic from the description and create short, simple, keyword-based queries.
        #                 - **GOOD QUERY EXAMPLE:** For a "Suibokuga" workshop, a good query is "warsztaty japońskiego malarstwa tuszem".
        #                 - **BAD QUERY EXAMPLE:** For a "C jak Cyrk" event, a bad query is "spektakl dla dzieci C jak Cyrk". This is bad because it uses the specific, unknown title. A better query would be "przedstawienie cyrkowe dla dzieci".

        #             **CRITICAL DATE-HANDLING LOGIC (You must follow this):**

        #             1.  **IF the event's start date is IN THE PAST** (before the current date):
        #                 - Your generated 'question' **MUST** include a specific date or timeframe that covers the event's date.
        #                    - e.g. "drugi tydzień czerwca", "w weekend 5-6 lipca", "w pierwszej połowie maja".
        #                 - **DO NOT** generate a generic, timeless query for a past event.

        #             2.  **IF the event's start date is IN THE FUTURE** (on or after the current date):
        #                 - You **CAN** generate a more generic query without a specific date, as the agent's default search window will find it.

        #             **EXAMPLE of Good vs. Bad Queries for a PAST event:**
        #             - Event Date: June 2, 2025. Current Date: July 18, 2025.
        #             - **BAD QUERY:** "warsztaty ceramiczne w Warszawie" (This is wrong because the user would not be interested in any past event).
        #             - **GOOD QUERY:** "warsztaty ceramiczne w Warszawie na początku czerwca" (This is correct because it forces the agent to search in the past).

        #             **EXAMPLE of Good vs. Bad Queries for a FUTURE event:**
        #             - Event Date: July 2, 2025. Current Date: June 18, 2025.
        #             - **GOOD QUERY:** "warsztaty ceramiczne w Warszawie" (This is fine because the agent will search from June 18 onwards and find the event).
        #             - **GOOD QUERY:** "Koncert rockowy w Lipcu w Warszawie" (This is fine because a user might be interested in an event in a specific timeframe).

        #             ---
        #             **ANSWER GENERATION RULES:**
        #             - The "answer" is the ground truth. It must contain the full, correct event details.
        #             - Use the provided Original Event Title for the title field.
        #             - Extract the date and location from the content.
        #             - **Original Event Title:** "{original_title}"

        #             **Output Format:**
        #             Return ONLY a JSON object matching this schema.
        #             {{
        #                 "questions": [
        #                     {{
        #                         "question": "The generated user search query.",
        #                         "answer": "Title: {original_title}\\nDate: [Comprehensive Date/Time Summary]\\nLocation: [Full Location]"
        #                     }}
        #                 ]
        #             }}
        #             """

        # prompt = f"""**Event Content to Analyze:**
        # {clean_content}

        # Current date is: {datetime.now().date().isoformat()}

        # ---
        # **Your Task: Generate the Single Most Likely Query**

        # Based on the event content, generate the **one single query** that an person, who does not know this event exists, would most likely type into a search engine.
        # ---
        # **CRITICAL INSTRUCTIONS FOR DATE-HANDLING LOGIC (YOU **MUST** FOLLOW THESE UNBREAKABLE GUIDELINES):**

        # 1.  **IF the event's start date is IN THE PAST** (before the current date):
        #     - Your generated 'question' **MUST** include a specific date or timeframe that covers the event's date (e.g., "drugi tydzień czerwca", "w weekend 5-6 lipca").

        # 2.  **IF the event's start date is IN THE FUTURE** (on or after the current date):
        #     - You **CAN** generate a more generic query without a specific date (e.g., "w lipcu", or no date at all).

        # **Guiding Principles for the Query:**

        # 1.  **Think Like the User, Not the Document:** The user does not know the event's specific title or jargon. Your query must reflect a real-world need.
        #     - **GOOD QUERY:** For a "Suibokuga" workshop, a good query is "warsztaty japońskiego malarstwa tuszem".
        #     - **BAD QUERY:** For a "C jak Cyrk" event, a bad query is "spektakl dla dzieci C jak Cyrk". A better query would be "przedstawienie cyrkowe dla dzieci".

        # 2.  **NEW: Prioritize Simplicity:** Avoid using overly specific or niche terms, even if they are in the document. Ask yourself: "Would my non-expert friend use these exact words?" If the answer is no, simplify the query to a more general category.
        #     - **Event Example:** A charity run for an oncology foundation.
        #     - **TOO SPECIFIC (Avoid this):** 'bieg charytatywny na rzecz onkologii'
        #     - **BETTER (Generate this):** 'bieg charytatywny warszawa' or 'maraton charytatywny'
        #     - **Event Example:** A festival of fantasy and board games.
        #     - **TOO SPECIFIC (Avoid this):** 'festiwal fantastyki i gier planszowych'
        #     - **BETTER (Generate this):** 'festiwal gier planszowych' or 'konwent fantastyki'

        # 3.  **Focus on Core Intent:** The query should clearly include the essential elements: the activity, the intended audience (if applicable), the general timeframe, and the location.


        # ---
        # **ANSWER GENERATION RULES:**

        # - The "answer" is the ground truth. It must contain the full, correct event details.
        # - Use the provided Original Event Title for the title field.
        # - Extract the date and location from the content.
        # - **Original Event Title:** "{original_title}"

        # ---
        # **Output Format:**
        # Return ONLY a JSON object matching this schema. Do not include any other text or explanations.
        # {{
        #     "questions": [
        #         {{
        #             "question": "The single, most likely user search query.",
        #             "answer": "Title: {original_title}\\nDate: [Comprehensive Date/Time Summary]\\nLocation: [Full Location]"
        #         }}
        #     ]
        # }}
        # """
#         prompt = f"""You are a highly precise data generation assistant. Your sole purpose is to follow the rules below to create a single JSON object from the provided text. Adherence to the rules is the only measure of success.

# ### CONTEXT
# **Event Content:**
# {clean_content}

# **Current Date:** {datetime.now().date().isoformat()}
# **Original Title:** "{original_title}"

# ### RULES
# You must follow these rules in order.

# **1. DATE LOGIC (MANDATORY FIRST STEP):**
# - **Analyze:** Find the event's start date in the context. Compare it to the Current Date.
# - **Apply Logic:**
#     - **If the event is in the PAST:** The `question` field in your JSON output **MUST** contain a specific timeframe (e.g., "w czerwcu 2025", "w ostatni weekend lipca").
#     - **Example of a past event (Date: 29.06.2025):**
#         - **INVALID QUERY:** `piknik rodzinny w Warszawie`
#         - **VALID QUERY:** `piknik rodzinny w Warszawie pod koniec czerwca`
#     - **If the event is in the FUTURE:** The `question` does not need a specific date.

# **2. QUERY SIMPLIFICATION (Secondary Guideline):**
# - **After** satisfying the date rule, simplify the query.
# - Do not use the original title or niche jargon. Focus on general terms a person might use.
#     - **GOOD:** `warsztaty japońskiego malarstwa tuszem`, `bieg charytatywny warszawa`
#     - **AVOID:** `warsztaty Suibokuga`, `bieg charytatywny na rzecz onkologii`

# ### OUTPUT FORMAT
# Return ONLY a single, valid JSON object. Do not include any other text, explanations, or markdown formatting around the JSON.
# ```json
# {{
#     "questions": [
#         {{
#             "question": "The single, rule-compliant user search query.",
#             "answer": "Title: {original_title}\\nDate: [Comprehensive Date/Time Summary]\\nLocation: [Full Location]"
#         }}
#     ]
# }}"""
#         system_prompt = "You are an expert Test Data Engineer. Your goal is to simulate a user searching for events by creating a single, realistic search query for the provided event content.. Return valid JSON only—no prose."
        prompt = f"""
        You are a world class expert AI engineer. Your role is to generate an evaluation dataset, consisting of query-answer pairs. Adherence to the outlined rules is the only measure of success.
        
        ### CONTEXT
        **Original Title:** 
        <original_title>"{original_title}"</original_title>

        <event_content>
        {clean_content}
        </event_content>

        **Current Date:** 
        <current_date>{datetime.now().date().isoformat()}</current_date>

        ### RULES
        You must follow each one of these rules in the following order.

        **1. DATE LOGIC (ABSOLUTELY CRUCIAL FIRST STEP):**
        - **Analyse:** Find the event's start date in the context
            - If the event context contains multiple dates, select the one closest to the current_date.
        - **Apply Logic:**
            - **If the event is in the PAST:** The 'question' field in your JSON output **MUST** contain a specific timeframe (e.g., "w na początku czerwca 2025", "w ostatni weekend lipca", "wakacje 2025"), making this particular event identifiable.
                - **Example of a past event (Date: 01.06.2025):**
                    - **INVALID QUESTION** 'piknik rodzinny w Warszawie'; 'piknik rodzinny w czerwcu'
                    - **VALID QUESTION** 'piknik rodzinny w warszawie na początku czerwca'; 'piknik rodzinny w warszawie pierwszego czerwca'
            - **If the event is in the FUTURE:** The 'question' does not require a specific timeframe.

        **2 QUESTION SIMPLIFICATION (Secondary Guideline):**
        - **After** satisfying the date rule, create a simple query.
        - Do no use the original title or niche jargon. Identify the underlying theme of the event, and focus on general terms a person might use.
            - **GOOD:** 'warsztaty japońskiego malarstwa tuszem', 'bieg charytatywny w Warszawie w sierpniu'
            - **BAD:** 'warsztaty Suibokuga', 'bieg charytatywny na rzecz onkologii'
        - **If an event relates to a specific type of activity**, it **MUST** be mentioned in the question.
            - **GOOD:** 'mecz piłkarski w Warszawie w Listopadzie', 'zajęcia pilates w czerwcu na Ursynowie'
            - **BAD:** 'wydarzenie sportowe w Warszawie w Listopadzie', 'zajęcia zdrowotne na Ursynowie'
        - **If an event relates to a specific entity**, the entity **MUST** be mentioned in the question.
            - **GOOD:** 'koncert zespołu Kult w Warszawie w wakacje', 'Mecz Legii w listpadzie'
            - **BAD:** 'koncert w Warszawie w wakacje', 'mecz piłkarski w listopadzie'
        - The question must be simple and straightforward (the question reader will not see the article OR title OR any other information about the event)


        ### OUTPUT FORMAT
        Return ONLY a single, valid JSON object. Do not include any other text, explanations, or markdown formatting around the JSON.
        ```json
        {{
            "questions": [
                {{
                    "question": "The single, rule-compliant user search query.",
                    "answer": "Title: {original_title}\\nDate: [Comprehensive Date/Time Summary]\\nLocation: [Full Location]"
                }}
            ]
        }}
        ```
        """
        response = await client.chat.completions.create(
            model=model,
            messages=[
                # {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt}
            ],
            response_format={"type": "json_object"},
            # temperature=0.2
            )
        try:
            response_content = response.choices[0].message.content
            if not response_content:
                return []
            response_json = json.loads(response_content)
            if isinstance(response_json, list):
                qa_pairs = response_json
            elif isinstance(response_json, dict):
                qa_pairs = response_json.get("pairs", response_json.get("questions", response_json.get("data", []))) 
                if not isinstance(qa_pairs, list):
                    return []
            else:
                qa_pairs = []
        except json.JSONDecodeError as e:
            print(f"JSON decode error: {e}")
            return []
        except Exception as e:
            print(f"Unexpected error: {e}")
            return []

        results = []
        for pair in qa_pairs:
            results.append({
                "question": pair["question"],
                "answer": pair["answer"],
                "filename": filename
            })
        
        return results

async def generate_random_questions(n_pages: int = 5, questions_per_page: int = 2) -> list[dict]:
    """
    Generate random questions from a set of event files
    """
    event_files = [f for f in os.listdir("./data/events") if f.endswith(".md")]
    selected_files = random.sample(event_files, min(n_pages, len(event_files)))
    tasks = []
    for filename in selected_files:
        filepath = os.path.join("./data/events", filename)
        tasks.append(generate_questions_for_file(filepath, questions_per_page))
    all_results = []
    with tqdm(total=len(tasks), desc="Generating questions") as pbar:
        for coro in asyncio.as_completed(tasks):
            try:
                result = await coro
                all_results.append(result)
                pbar.update(1)
            except Exception as e:
                pbar.update(1)
                continue

    all_questions = []
    for questions in all_results:
        all_questions.extend(questions)
    return all_questions

async def main():
    n_pages = 10
    questions = await generate_random_questions(n_pages=n_pages, questions_per_page=1)
    print(f"\nGenerated {len(questions)} total questions")

    for i, q in enumerate(questions): 
        print(f"\n{i+1}. Q: {q['question']}")
        print(f"   A: {q['answer']}")
        print(f"   File: {q['filename']}")
    
    return questions

questions = await main()

Generating questions:   0%|          | 0/10 [00:00<?, ?it/s]


Generated 10 total questions

1. Q: turniej siatkówki plażowej dla kobiet
   A: Title: Volleyball World Beach Pro Tour Futures Women
Date: 11.09.2025 - 14.09.2025
Location: Plaża Wilanów
   File: volleyball_world_beach_pro_tour_futures_women_00068.md

2. Q: posiedzenie komisji rewizyjnej w Rembertowie w połowie lipca 2025
   A: Title: Komisja Rewizyjna
Date: 15.07.2025, 17:00
Location: Urząd Dzielnicy Rembertów m.st. Warszawy, al. gen. A. Chruściela 'Montera' 28, 04-401 Warszawa, 04-401 Warszawa
   File: komisja_rewizyjna_00114.md

3. Q: koncert Janka Młynarskiego w Rembertowie pod koniec czerwca
   A: Title: Koncert - Janek Młynarski i Warszawskie Combo Tanceczne
Date: 29.06.2025 19:00 - 29.06.2025 21:00
Location: Estrada Domu Kultury "Rembertów", Komandosów 8, 04-485, Warszawa
   File: koncert_janek_młynarski_i_warszawskie_combo_tanceczne_00142.md

4. Q: festiwal dożynkowy w Warszawie
   A: Title: Powśińskie Dożynki
Date: 24.08.2025
Location: Plenery Domu Pracy Twórczej, Ptysiowa 3,

In [13]:
model

'gpt-4o'

In [14]:
filename=f"./synth_datasets/raw_queries/generated_QUERIES_Polish_{timestamp}_{model}.json"
filename

'./synth_datasets/raw_queries/generated_QUERIES_Polish_20250721_1450_gpt-4o.json'

In [15]:
with open(filename, "w", encoding="utf-8") as f:
    json.dump(questions, f, indent=2, ensure_ascii=False)